# Research Notebook 06: Quantitative Market Risk & VaR Backtesting
## Value at Risk, Expected Shortfall, Kupiec POF & Christoffersen Tests

### 1. Risk Measures
- **Value at Risk (VaR):** Exceedance threshold quantile.
- **Expected Shortfall (CVaR):** $ES_\alpha = E[L \mid L > VaR_\alpha]$.
- **Kupiec (1995) POF Likelihood Ratio Test:** Unconditional coverage $LR_{pof} \sim \chi^2(1)$.
- **Christoffersen (1998) Independence Test:** No violation clustering $LR_{ind} \sim \chi^2(1)$.


In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from volatility_lab.data.providers.synthetic import SyntheticMarketDataProvider
from volatility_lab.data.preprocessing import DataPreprocessor
from volatility_lab.risk.var import ValueAtRisk
from volatility_lab.risk.expected_shortfall import ExpectedShortfall
from volatility_lab.risk.backtesting import VaRBacktester

raw = SyntheticMarketDataProvider(seed=42).fetch_data('NIFTY50', '2020-01-01', '2023-12-31')
df = DataPreprocessor().clean_and_prepare(raw)
returns = df['log_return']

var_calc = ValueAtRisk(confidence_levels=[0.95, 0.99])
var_norm = var_calc.parametric_normal_var(returns)
var_t = var_calc.parametric_student_t_var(returns)
var_hist = var_calc.historical_var(returns)

es_calc = ExpectedShortfall(confidence_levels=[0.95])
es_norm = es_calc.parametric_normal_es(returns)

# Backtest 95% Parametric Normal VaR
v95 = var_norm['var_norm_95'].dropna()
bt = VaRBacktester.backtest(returns.reindex(v95.index), v95, confidence_level=0.95)

print(f"Total Observations: {bt.total_observations}")
print(f"Actual Violations: {bt.actual_violations} (Expected: {bt.expected_violations:.1f})")
print(f"Violation Ratio: {bt.violation_ratio:.2f}")
print(f"Kupiec POF LR: {bt.kupiec_lr_stat:.3f} (p={bt.kupiec_p_value:.4f}) => {bt.kupiec_decision}")
print(f"Christoffersen Independence LR: {bt.christoffersen_ind_stat:.3f} (p={bt.christoffersen_ind_p_value:.4f}) => {bt.christoffersen_ind_decision}")


### 2. Regulatory Implications & Basel Traffic Light System
- Normal VaR systematically underestimates tail losses during market crashes (fat tails).
- Student-t VaR and GARCH-filtered VaR pass conditional coverage tests by dynamically expanding during stress periods.
